# Sentiment Analysis: Sparse vs Dense Representations

This notebook compares **six document representation strategies** for binary sentiment classification on the IMDB movie review dataset.

| Category | Method | Representation |
|---|---|---|
| Sparse | **Bag-of-Words** | Raw term counts |
| Sparse | **TF-IDF** | Count × inverse doc frequency |
| Sparse | **BM25** | Saturated TF × smoothed IDF |
| Dense | **Mean pooling** | Average of word vectors |
| Dense | **TF-IDF weighted mean** | IDF-weighted average of word vectors |
| Dense | **Min/Mean/Max pooling** | Concatenation of three pooling ops |

All six feed into the same **Logistic Regression** classifier — the only thing that changes is the feature representation.

A **zero-shot baseline** (no classifier, just cosine similarity to seed words) is included at the end.

In [ ]:
!pip install -q datasets gensim scikit-learn rank_bm25 matplotlib seaborn transformers torch

In [ ]:
import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

from datasets import load_dataset

from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import normalize
from scipy.sparse import issparse

from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

plt.style.use('ggplot')
%matplotlib inline
print('Ready.')

---
## 1. Data — IMDB Movie Reviews

50,000 movie reviews labeled **positive** (1) or **negative** (0).  
We use 10,000 for training and 5,000 for testing to keep runtimes short.

In [ ]:
print('Loading IMDB...')
raw = load_dataset('imdb')

# Subsample for speed
N_TRAIN, N_TEST = 10_000, 5_000
train_data = raw['train'].shuffle(seed=42).select(range(N_TRAIN))
test_data  = raw['test'].shuffle(seed=42).select(range(N_TEST))

train_texts  = train_data['text']
train_labels = train_data['label']
test_texts   = test_data['text']
test_labels  = test_data['label']

print(f'Train: {len(train_texts):,}  |  Test: {len(test_texts):,}')
print(f'Label balance (train): {sum(train_labels)/len(train_labels):.2f} positive')
print(f'\nSample review:\n{train_texts[0][:300]}...')

In [ ]:
def clean(text):
    """Remove HTML tags and collapse whitespace."""
    text = re.sub(r'<[^>]+>', ' ', text)   # strip HTML
    text = re.sub(r'\s+', ' ', text)        # collapse whitespace
    return text.strip().lower()

train_clean = [clean(t) for t in train_texts]
test_clean  = [clean(t) for t in test_texts]

# Tokenized version for Word2Vec training
train_tokens = [simple_preprocess(t) for t in train_clean]
test_tokens  = [simple_preprocess(t) for t in test_clean]

print('Sample cleaned + tokenized:')
print(' '.join(train_tokens[0][:20]), '...')

---
## 2. Sparse Representations

### 2.1 Bag-of-Words

Each document is a vector of raw **term counts** over the vocabulary.  
Completely ignores word order and document length.

In [ ]:
bow_vec = CountVectorizer(max_features=30_000, min_df=3)
X_train_bow = bow_vec.fit_transform(train_clean)
X_test_bow  = bow_vec.transform(test_clean)

print(f'BoW matrix shape: {X_train_bow.shape}')
print(f'Sparsity: {1 - X_train_bow.nnz / (X_train_bow.shape[0]*X_train_bow.shape[1]):.4f}')

### 2.2 TF-IDF

Weights each term by how **rare** it is across the corpus — common words like "the" get down-weighted.  

$$\text{TF-IDF}(t, d) = \text{tf}(t,d) \times \log\frac{N}{\text{df}(t)}$$

In [ ]:
tfidf_vec = TfidfVectorizer(max_features=30_000, min_df=3, sublinear_tf=True)
X_train_tfidf = tfidf_vec.fit_transform(train_clean)
X_test_tfidf  = tfidf_vec.transform(test_clean)

print(f'TF-IDF matrix shape: {X_train_tfidf.shape}')

### 2.3 BM25

BM25 (Best Match 25) improves on TF-IDF in two ways:

1. **Term frequency saturation** — doubling a word's count doesn't double its score (controlled by `k1`)
2. **Length normalization** — penalizes long documents that just repeat words (controlled by `b`)

$$\text{BM25}(t, d) = \text{IDF}(t) \cdot \frac{\text{tf}(t,d) \cdot (k_1 + 1)}{\text{tf}(t,d) + k_1 \cdot \left(1 - b + b \cdot \frac{|d|}{\text{avgdl}}\right)}$$

Standard parameters: `k1 = 1.5`, `b = 0.75`

In [ ]:
import scipy.sparse as sp

class BM25Transformer:
    """
    Transforms a raw-count matrix (from CountVectorizer) into BM25-weighted features.
    Follows the Okapi BM25 formula.
    """
    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1
        self.b  = b

    def fit(self, X):
        # X: sparse count matrix (n_docs × vocab)
        n_docs = X.shape[0]
        self.avgdl_ = X.sum(axis=1).mean()           # average document length
        df = (X > 0).sum(axis=0).A1                  # document frequency per term
        # Smoothed IDF (Robertson & Zaragoza)
        self.idf_ = np.log((n_docs - df + 0.5) / (df + 0.5) + 1.0)
        return self

    def transform(self, X):
        X = sp.csr_matrix(X, dtype=np.float64)
        dl = X.sum(axis=1).A1                         # document lengths
        # BM25 TF: tf*(k1+1) / (tf + k1*(1-b+b*dl/avgdl))
        denom_diag = sp.diags(
            1.0 / (self.k1 * (1 - self.b + self.b * dl / self.avgdl_))
        )
        # Numerator: tf * (k1+1)
        tf_num = X.multiply(self.k1 + 1)
        # Denominator: tf + k1*(1-b+b*dl/avgdl)  per row
        # = X + diag(k1*(1-b+b*dl/avgdl)) * ones
        row_norm = self.k1 * (1 - self.b + self.b * dl / self.avgdl_)  # (n_docs,)
        # For each non-zero entry (i,j): tf_ij / (tf_ij + row_norm_i)
        X_coo = X.tocoo()
        rows, cols, vals = X_coo.row, X_coo.col, X_coo.data
        bm25_tf = (vals * (self.k1 + 1)) / (vals + row_norm[rows])
        bm25_matrix = sp.csr_matrix(
            (bm25_tf, (rows, cols)), shape=X.shape
        )
        # Multiply by IDF
        return bm25_matrix.multiply(self.idf_)

    def fit_transform(self, X):
        return self.fit(X).transform(X)


# Use the same CountVectorizer vocab for a fair comparison
bm25_transformer = BM25Transformer(k1=1.5, b=0.75)
X_train_bm25 = bm25_transformer.fit_transform(X_train_bow)
X_test_bm25  = bm25_transformer.transform(X_test_bow)

print(f'BM25 matrix shape: {X_train_bm25.shape}')

# Quick sanity: compare TF-IDF vs BM25 weights for the same term in a sample doc
sample_idx = 0
vocab = bow_vec.get_feature_names_out()
tfidf_row = X_train_tfidf[sample_idx].toarray().ravel()
bm25_row  = X_train_bm25[sample_idx].toarray().ravel()
top_terms = tfidf_row.argsort()[-8:][::-1]

print(f'\nTop terms in review #{sample_idx} — TF-IDF vs BM25 weight:')
print(f"{'Term':20s}  {'TF-IDF':8s}  {'BM25':8s}")
for i in top_terms:
    print(f"{vocab[i]:20s}  {tfidf_row[i]:.4f}    {bm25_row[i]:.4f}")

---
## 3. Dense Representations (Word2Vec)

First we train a Word2Vec model **on the IMDB corpus itself** — domain-specific vectors capture sentiment-relevant context better than general-purpose vectors (e.g., Wikipedia-trained).

In [ ]:
print('Training Word2Vec on IMDB corpus...')
start = time.time()

w2v = Word2Vec(
    sentences=train_tokens,
    vector_size=100,
    window=5,
    min_count=3,
    sg=1,           # Skip-gram
    negative=5,
    epochs=10,
    seed=42,
    workers=4,
)

print(f'Done in {time.time()-start:.1f}s')
print(f'Vocabulary: {len(w2v.wv):,} words')

# Quick sanity — sentiment words should cluster
print('\nNearest to "excellent":', [w for w,_ in w2v.wv.most_similar('excellent', topn=5)])
print('Nearest to "terrible":', [w for w,_ in w2v.wv.most_similar('terrible', topn=5)])

### 3.1 Mean Pooling

Average all word vectors in a document. Simple but loses word order and treats all words equally.

In [ ]:
def mean_pool(tokens, wv):
    vecs = [wv[w] for w in tokens if w in wv]
    if not vecs:
        return np.zeros(wv.vector_size)
    return np.mean(vecs, axis=0)

X_train_mean = np.array([mean_pool(t, w2v.wv) for t in train_tokens])
X_test_mean  = np.array([mean_pool(t, w2v.wv) for t in test_tokens])

print(f'Mean pool shape: {X_train_mean.shape}')

### 3.2 TF-IDF Weighted Mean

Weight each word vector by its TF-IDF score before averaging — rare, informative words contribute more than common ones like "the" or "is".

In [ ]:
# Build word → IDF weight lookup from the already-fitted TF-IDF vectorizer
idf_lookup = dict(zip(tfidf_vec.get_feature_names_out(), tfidf_vec.idf_))

def weighted_mean_pool(tokens, wv, idf_lookup):
    vecs, weights = [], []
    for w in tokens:
        if w in wv:
            vecs.append(wv[w])
            weights.append(idf_lookup.get(w, 1.0))  # default weight 1 for OOV
    if not vecs:
        return np.zeros(wv.vector_size)
    weights = np.array(weights)
    return np.average(vecs, axis=0, weights=weights)

X_train_wt = np.array([weighted_mean_pool(t, w2v.wv, idf_lookup) for t in train_tokens])
X_test_wt  = np.array([weighted_mean_pool(t, w2v.wv, idf_lookup) for t in test_tokens])

print(f'Weighted mean shape: {X_train_wt.shape}')

### 3.3 Min / Mean / Max Pooling

Concatenate three pooling operations over the word vectors:

- **Mean**: overall semantic content
- **Max**: strongest signal per dimension (picks up on the most extreme/salient words)
- **Min**: captures negations and weak signals

Result: a `3 × vector_size` = 300-dimensional vector.

In [ ]:
def minmeanmax_pool(tokens, wv):
    vecs = [wv[w] for w in tokens if w in wv]
    if not vecs:
        return np.zeros(wv.vector_size * 3)
    mat = np.array(vecs)
    return np.concatenate([mat.mean(0), mat.max(0), mat.min(0)])

X_train_mmm = np.array([minmeanmax_pool(t, w2v.wv) for t in train_tokens])
X_test_mmm  = np.array([minmeanmax_pool(t, w2v.wv) for t in test_tokens])

print(f'Min/Mean/Max shape: {X_train_mmm.shape}  (3 × {w2v.wv.vector_size})')

---
### 3.4 BERT Input Embeddings (Static)

A transformer model is two separate stages:

```
token IDs  →  [Input embedding matrix]  →  static vectors    ← THIS section
              shape: (vocab=30522, 768)      no context, just lookup

static vecs →  [12 × Self-Attention layers]  →  contextual vectors  ← next section
```

The **input embedding matrix** (`model.embeddings.word_embeddings.weight`) is a fixed lookup table — one 768-dim vector per WordPiece token. It is:
- **Static** — same token always produces the same vector, like Word2Vec
- Trained via **masked language model** backprop (not skip-gram)
- **Subword** tokenisation — "running" → `["running"]` or `["run", "##ning"]`

Mean-pooling these static subword vectors over a sentence gives a representation that is **directly comparable to Word2Vec mean pooling**, with two differences: higher dimension (768 vs 100) and a different training signal.

In [ ]:
import torch
from transformers import BertTokenizer, BertModel

print("Loading bert-base-uncased...")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert = BertModel.from_pretrained('bert-base-uncased')
bert.eval()

# ── The input embedding matrix ──────────────────────────────────────────────
# This is a plain nn.Embedding: (vocab_size, hidden_size) = (30522, 768)
# It is the ONLY part of BERT that is used here — no attention, no layers.
input_emb_matrix = bert.embeddings.word_embeddings.weight.detach().numpy()

print(f"Input embedding matrix shape: {input_emb_matrix.shape}")
print(f"  vocab_size  = {input_emb_matrix.shape[0]:,}  (WordPiece tokens)")
print(f"  hidden_size = {input_emb_matrix.shape[1]}  (dimensions per token)")

# Compare: how is the same word tokenised differently in W2V vs BERT?
demo_words = ['running', 'unbelievable', 'masterpiece', 'disappointment']
print("\nWord2Vec vs BERT tokenisation:")
print(f"  {'Word':20s}  {'W2V (1 token)':15s}  BERT subword tokens")
print("  " + "-" * 60)
for w in demo_words:
    bert_tokens = tokenizer.tokenize(w)
    in_w2v = w in w2v.wv
    print(f"  {w:20s}  {'✓' if in_w2v else '✗ OOV':15s}  {bert_tokens}")

In [ ]:
def bert_input_mean_pool(text, tokenizer, emb_matrix, max_len=128):
    """
    Mean-pool BERT INPUT (static) embeddings over subword tokens.
    Excludes [CLS] and [SEP] special tokens.
    No BERT forward pass — pure matrix lookup.
    """
    ids = tokenizer.encode(
        text,
        max_length=max_len,
        truncation=True,
        add_special_tokens=False,   # skip [CLS]=101, [SEP]=102
    )
    if not ids:
        return np.zeros(emb_matrix.shape[1])
    return emb_matrix[ids].mean(axis=0)


print("Building BERT input embedding features (no forward pass)...")
start = time.time()

X_train_bert_input = np.array([
    bert_input_mean_pool(t, tokenizer, input_emb_matrix) for t in train_clean
])
X_test_bert_input = np.array([
    bert_input_mean_pool(t, tokenizer, input_emb_matrix) for t in test_clean
])

print(f"Done in {time.time()-start:.1f}s")
print(f"Shape: {X_train_bert_input.shape}  (mean over subword tokens, 768-dim)")

### 3.5 BERT [CLS] Contextual Embeddings

Now we run the **full BERT forward pass** — all 12 attention layers.  
The `[CLS]` token at position 0 is trained to aggregate sentence-level meaning and is the standard representation used for classification.

This is **not static** — the same word produces a different vector depending on all surrounding words.

In [ ]:
def get_cls_embeddings(texts, model, tokenizer, batch_size=64, max_len=128):
    """Run BERT forward pass and return the [CLS] token vector for each text."""
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = model.to(device)
    all_cls = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        inputs = tokenizer(
            batch,
            max_length=max_len,
            truncation=True,
            padding=True,
            return_tensors='pt',
        ).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
        # last_hidden_state[:, 0, :] is the [CLS] token output
        cls_vecs = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_cls.append(cls_vecs)

        if (i // batch_size) % 10 == 0:
            print(f"  {i + len(batch):>6,} / {len(texts):,}", end='\r')

    return np.vstack(all_cls)


print(f"Running BERT forward pass on {len(train_clean)+len(test_clean):,} texts...")
print("(CPU: ~5–10 min  |  GPU: ~1 min)\n")
start = time.time()

X_train_cls = get_cls_embeddings(train_clean, bert, tokenizer)
X_test_cls  = get_cls_embeddings(test_clean,  bert, tokenizer)

print(f"\nDone in {time.time()-start:.0f}s")
print(f"Shape: {X_train_cls.shape}  ([CLS] contextual vector, 768-dim)")

---
## 4. Train & Evaluate All Six Methods

Same classifier (Logistic Regression) for all — only the features differ.

In [ ]:
def evaluate(name, X_train, X_test, y_train, y_test):
    clf = LogisticRegression(max_iter=1000, C=1.0, solver='saga', n_jobs=-1)
    t0 = time.time()
    clf.fit(X_train, y_train)
    train_time = time.time() - t0

    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred)
    dim = X_train.shape[1]

    print(f"{name:35s}  acc={acc:.4f}  f1={f1:.4f}  dim={dim:6,}  time={train_time:.1f}s")
    return {'name': name, 'accuracy': acc, 'f1': f1, 'dim': dim, 'time': train_time}


experiments = [
    # Sparse
    ('Bag-of-Words',               X_train_bow,         X_test_bow),
    ('TF-IDF',                     X_train_tfidf,       X_test_tfidf),
    ('BM25',                       X_train_bm25,        X_test_bm25),
    # Dense — Word2Vec
    ('W2V Mean',                   X_train_mean,        X_test_mean),
    ('W2V TF-IDF Weighted',        X_train_wt,          X_test_wt),
    ('W2V Min/Mean/Max',           X_train_mmm,         X_test_mmm),
    # Dense — BERT static (input matrix only, no attention)
    ('BERT Input Embeddings',      X_train_bert_input,  X_test_bert_input),
    # Dense — BERT contextual ([CLS] after 12 attention layers)
    ('BERT [CLS] Contextual',      X_train_cls,         X_test_cls),
]

print(f"{'Method':35s}  {'Accuracy':8s}  {'F1':6s}  {'Dim':8s}  {'Fit time'}")
print('-' * 78)

results = []
for name, X_tr, X_te in experiments:
    r = evaluate(name, X_tr, X_te, train_labels, test_labels)
    results.append(r)

results_df = pd.DataFrame(results).sort_values('accuracy', ascending=False)
print(f"\nBest:  {results_df.iloc[0]['name']} ({results_df.iloc[0]['accuracy']:.4f})")
print(f"Worst: {results_df.iloc[-1]['name']} ({results_df.iloc[-1]['accuracy']:.4f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

def method_color(name):
    if name in {'Bag-of-Words', 'TF-IDF', 'BM25'}:
        return '#3498db'   # blue  — sparse
    if 'W2V' in name:
        return '#e74c3c'   # red   — Word2Vec dense
    if 'Input' in name:
        return '#f39c12'   # orange — BERT static (no attention)
    if 'CLS' in name:
        return '#2ecc71'   # green  — BERT contextual
    return '#95a5a6'

sorted_results = sorted(results, key=lambda x: x['accuracy'])
bar_colors = [method_color(r['name']) for r in sorted_results]
names = [r['name'] for r in sorted_results]
accs  = [r['accuracy'] for r in sorted_results]
f1s   = [r['f1']  for r in sorted_results]

# Accuracy
bars = axes[0].barh(names, accs, color=bar_colors, alpha=0.87)
axes[0].set_xlim(0.70, 1.0)
axes[0].set_xlabel('Test Accuracy')
axes[0].set_title('Accuracy by Representation')
for bar, acc in zip(bars, accs):
    axes[0].text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
                 f'{acc:.4f}', va='center', fontsize=9)

# F1
bars = axes[1].barh(names, f1s, color=bar_colors, alpha=0.87)
axes[1].set_xlim(0.70, 1.0)
axes[1].set_xlabel('Test F1 Score')
axes[1].set_title('F1 by Representation')
for bar, f1 in zip(bars, f1s):
    axes[1].text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
                 f'{f1:.4f}', va='center', fontsize=9)

from matplotlib.patches import Patch
legend = [
    Patch(color='#3498db', label='Sparse (BoW / TF-IDF / BM25)'),
    Patch(color='#e74c3c', label='Dense — Word2Vec'),
    Patch(color='#f39c12', label='Dense — BERT input matrix (static)'),
    Patch(color='#2ecc71', label='Dense — BERT [CLS] (contextual)'),
]
axes[0].legend(handles=legend, loc='lower right', fontsize=8)

plt.suptitle('Sentiment Classification — Representation Comparison (IMDB)', fontsize=13)
plt.tight_layout()
plt.show()

# ── Key numbers table ──────────────────────────────────────────────────────
print(f"\n{'Method':35s}  {'Acc':>7s}  {'Dim':>7s}  Notes")
print("-" * 75)
for r in sorted(results, key=lambda x: -x['accuracy']):
    notes = ''
    if 'Input' in r['name']:
        notes = 'static — no attention'
    elif 'CLS' in r['name']:
        notes = 'contextual — 12 attention layers'
    elif 'W2V' in r['name']:
        notes = 'static — skip-gram trained'
    elif r['name'] in {'Bag-of-Words','TF-IDF','BM25'}:
        notes = f"sparse {r['dim']:,}-dim"
    print(f"{r['name']:35s}  {r['accuracy']:>7.4f}  {r['dim']:>7,}  {notes}")

---
## 5. Why Do Sparse Methods Beat Dense on Sentiment?

This surprises many people. The reason is structural:

**Word2Vec encodes *distributional similarity*, not *sentiment opposition*.**  
Words that are antonyms ("good" / "bad", "love" / "hate") appear in almost identical contexts → they end up *close* in vector space.

Sparse methods keep these words as **separate dimensions** — the model can learn independent weights for "love" and "hate".

In [ ]:
# Demonstrate: antonyms are close in Word2Vec space
antonym_pairs = [
    ('good',      'bad'),
    ('love',      'hate'),
    ('excellent', 'terrible'),
    ('amazing',   'awful'),
    ('brilliant', 'horrible'),
    ('enjoyed',   'hated'),
]

print('Cosine similarity between sentiment antonyms:')
print(f"  {'Word A':12s}  {'Word B':12s}  {'Similarity':>10s}")
print('  ' + '-' * 40)
for w1, w2 in antonym_pairs:
    if w1 in w2v.wv and w2 in w2v.wv:
        sim = w2v.wv.similarity(w1, w2)
        print(f"  {w1:12s}  {w2:12s}  {sim:10.4f}")

print('\nHigh similarity between antonyms = they encode "reviews about quality"')
print('not "positive quality" vs "negative quality".')

In [ ]:
# Demonstrate the negation problem with mean pooling
# "not good" and "good" produce nearly identical mean vectors

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)

pairs = [
    ("this movie was absolutely terrible and awful",
     "this movie was absolutely wonderful and great"),
    ("i did not enjoy this film at all",
     "i did enjoy this film a lot"),
    ("the acting was not good",
     "the acting was good"),
    ("i would not recommend this",
     "i would recommend this"),
]

print('Similarity between opposite-sentiment pairs (mean pooling):')
print(f"  {'Similarity':>10s}  Pair")
print('  ' + '-' * 70)
for neg, pos in pairs:
    v_neg = mean_pool(simple_preprocess(neg), w2v.wv)
    v_pos = mean_pool(simple_preprocess(pos), w2v.wv)
    sim = cosine_sim(v_neg, v_pos)
    print(f"  {sim:10.4f}  '{neg[:40]}' vs")
    print(f"  {'':10s}  '{pos[:40]}'")

print('\nHigh cosine similarity between opposite-sentiment sentences!')
print('Mean pooling cannot distinguish negation — "not good" ≈ "good".')

---
## 6. Error Analysis

Where does each approach fail? Let's compare the **errors made by TF-IDF vs W2V Min/Mean/Max** to understand what each method captures and misses.

In [ ]:
# Retrain both classifiers and collect predictions
clf_tfidf = LogisticRegression(max_iter=1000, C=1.0, solver='saga', n_jobs=-1)
clf_tfidf.fit(X_train_tfidf, train_labels)
pred_tfidf = clf_tfidf.predict(X_test_tfidf)

clf_mmm = LogisticRegression(max_iter=1000, C=1.0, solver='saga', n_jobs=-1)
clf_mmm.fit(X_train_mmm, train_labels)
pred_mmm = clf_mmm.predict(X_test_mmm)

y_true = np.array(test_labels)

# Error categories
both_wrong   = (pred_tfidf != y_true) & (pred_mmm != y_true)
tfidf_only   = (pred_tfidf != y_true) & (pred_mmm == y_true)
w2v_only     = (pred_tfidf == y_true) & (pred_mmm != y_true)
both_correct = (pred_tfidf == y_true) & (pred_mmm == y_true)

print(f'Both correct:        {both_correct.sum():5d} ({100*both_correct.mean():.1f}%)')
print(f'Both wrong:          {both_wrong.sum():5d} ({100*both_wrong.mean():.1f}%)')
print(f'Only TF-IDF wrong:   {tfidf_only.sum():5d} ({100*tfidf_only.mean():.1f}%)')
print(f'Only W2V wrong:      {w2v_only.sum():5d} ({100*w2v_only.mean():.1f}%)')

# Show examples where W2V is right but TF-IDF is wrong
# These are cases where semantic generalisation helped
w2v_wins_idx = np.where(tfidf_only)[0][:3]
print('\n--- Cases where W2V is right but TF-IDF is wrong (semantic generalisation) ---')
for idx in w2v_wins_idx:
    label = 'POS' if y_true[idx] == 1 else 'NEG'
    print(f'[{label}] {test_clean[idx][:200]}\n')

# Show examples where TF-IDF is right but W2V is wrong
# These are likely negation or rare vocabulary cases
tfidf_wins_idx = np.where(w2v_only)[0][:3]
print('--- Cases where TF-IDF is right but W2V is wrong (negation / specific terms) ---')
for idx in tfidf_wins_idx:
    label = 'POS' if y_true[idx] == 1 else 'NEG'
    print(f'[{label}] {test_clean[idx][:200]}\n')

In [ ]:
# What are the most predictive words for TF-IDF + LogReg?
feature_names = tfidf_vec.get_feature_names_out()
coefs = clf_tfidf.coef_[0]

top_pos_idx = coefs.argsort()[-20:][::-1]
top_neg_idx = coefs.argsort()[:20]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh([feature_names[i] for i in top_pos_idx[::-1]],
             coefs[top_pos_idx[::-1]], color='#2ecc71', alpha=0.85)
axes[0].set_title('Top 20 Positive Sentiment Words (TF-IDF coefficients)')
axes[0].set_xlabel('Logistic Regression coefficient')

axes[1].barh([feature_names[i] for i in top_neg_idx],
             coefs[top_neg_idx], color='#e74c3c', alpha=0.85)
axes[1].set_title('Top 20 Negative Sentiment Words')
axes[1].set_xlabel('Logistic Regression coefficient')

plt.tight_layout()
plt.show()
print('These are the exact words driving predictions — fully interpretable.')

---
## 7. Bonus: Zero-Shot Classification via Cosine Similarity

Can we classify sentiment **without any labeled training data**, using only the geometry of word vectors?

Three strategies, from naive to principled:

In [ ]:
# Strategy 1: cosine similarity to the single words "positive" and "negative"
# Hypothesis: a positive review vector should be closer to "positive" than to "negative"

if 'positive' in w2v.wv and 'negative' in w2v.wv:
    vec_pos = w2v.wv['positive']
    vec_neg = w2v.wv['negative']

    # Check what these words are actually close to
    print('Nearest neighbours of "positive":')
    print(' ', [w for w,_ in w2v.wv.most_similar('positive', topn=8)])
    print('Nearest neighbours of "negative":')
    print(' ', [w for w,_ in w2v.wv.most_similar('negative', topn=8)])
    print(f'\nSimilarity(positive, negative) = {w2v.wv.similarity("positive","negative"):.4f}')
    print('→ They are close! Both appear in contexts like "test result" or "feedback".')

    preds_naive = []
    for tokens in test_tokens:
        rv = mean_pool(tokens, w2v.wv)
        sim_p = cosine_sim(rv, vec_pos)
        sim_n = cosine_sim(rv, vec_neg)
        preds_naive.append(1 if sim_p > sim_n else 0)

    acc_naive = accuracy_score(test_labels, preds_naive)
    print(f'\nZero-shot accuracy ("positive" vs "negative"): {acc_naive:.4f}')
    print('(~0.50–0.60 — barely above random)')
else:
    print('"positive"/"negative" not in vocabulary — corpus too domain-specific.')

In [ ]:
# Strategy 2: richer seed word clusters
# Use the mean of several clearly positive / negative words as anchors

pos_seeds = ['excellent', 'wonderful', 'brilliant', 'amazing', 'loved',
             'outstanding', 'superb', 'enjoyable', 'fantastic', 'masterpiece']
neg_seeds = ['terrible', 'awful', 'horrible', 'boring', 'waste',
             'disappointing', 'dreadful', 'pathetic', 'unwatchable', 'worst']

pos_seeds = [w for w in pos_seeds if w in w2v.wv]
neg_seeds = [w for w in neg_seeds if w in w2v.wv]

anchor_pos = np.mean([w2v.wv[w] for w in pos_seeds], axis=0)
anchor_neg = np.mean([w2v.wv[w] for w in neg_seeds], axis=0)

print(f'Positive seed words ({len(pos_seeds)}): {pos_seeds}')
print(f'Negative seed words ({len(neg_seeds)}): {neg_seeds}')
print(f'Similarity between anchors: {cosine_sim(anchor_pos, anchor_neg):.4f}')
print('(Should be lower than "positive" vs "negative" — better separated)')

preds_seeds = []
for tokens in test_tokens:
    rv = mean_pool(tokens, w2v.wv)
    sim_p = cosine_sim(rv, anchor_pos)
    sim_n = cosine_sim(rv, anchor_neg)
    preds_seeds.append(1 if sim_p > sim_n else 0)

acc_seeds = accuracy_score(test_labels, preds_seeds)
print(f'\nZero-shot accuracy (seed word clusters): {acc_seeds:.4f}')

In [ ]:
# Strategy 3: sentiment axis — difference of known antonym pairs
# Build a direction in vector space that points from negative → positive

antonym_pairs_axis = [
    ('excellent', 'terrible'),
    ('wonderful', 'awful'),
    ('loved',     'hated'),
    ('amazing',   'horrible'),
    ('brilliant', 'dreadful'),
    ('great',     'bad'),
    ('good',      'poor'),
]
valid_pairs = [(p, n) for p, n in antonym_pairs_axis
               if p in w2v.wv and n in w2v.wv]

# Sentiment axis = mean of (positive_word - negative_word) directions
sentiment_axis = np.mean(
    [w2v.wv[p] - w2v.wv[n] for p, n in valid_pairs], axis=0
)
sentiment_axis /= np.linalg.norm(sentiment_axis)  # unit vector

# Classify: project review vector onto sentiment axis
# Positive dot product → positive sentiment
scores = np.array([mean_pool(t, w2v.wv) for t in test_tokens]) @ sentiment_axis
preds_axis = (scores > 0).astype(int)

acc_axis = accuracy_score(test_labels, preds_axis)
print(f'Antonym pairs used: {valid_pairs}')
print(f'\nZero-shot accuracy (sentiment axis): {acc_axis:.4f}')

# Sanity: project the seed words themselves onto the axis
print('\nProjection of words onto sentiment axis (positive = good sentiment):')
check_words = pos_seeds[:5] + neg_seeds[:5]
for w in check_words:
    if w in w2v.wv:
        proj = np.dot(w2v.wv[w], sentiment_axis)
        bar = '█' * int(abs(proj) * 20)
        sign = '+' if proj > 0 else '-'
        print(f"  {w:15s} {sign}{abs(proj):.3f}  {bar}")

In [ ]:
# Final comparison: all approaches including zero-shot
all_results = results + [
    {'name': 'Zero-shot: "pos"/"neg"',   'accuracy': accuracy_score(test_labels, preds_naive) if 'positive' in w2v.wv else 0.5, 'f1': f1_score(test_labels, preds_naive) if 'positive' in w2v.wv else 0.5},
    {'name': 'Zero-shot: seed clusters', 'accuracy': acc_seeds, 'f1': f1_score(test_labels, preds_seeds)},
    {'name': 'Zero-shot: sentiment axis','accuracy': acc_axis,  'f1': f1_score(test_labels, preds_axis)},
]

all_results_df = pd.DataFrame(all_results).sort_values('accuracy')

def get_color(name):
    if 'Zero-shot' in name: return '#95a5a6'
    if name in sparse_names: return '#3498db'
    return '#e74c3c'

bar_colors = [get_color(r['name']) for r in all_results_df.to_dict('records')]

fig, ax = plt.subplots(figsize=(11, 7))
bars = ax.barh(all_results_df['name'], all_results_df['accuracy'],
               color=bar_colors, alpha=0.87)
ax.axvline(x=0.5, color='black', linestyle=':', linewidth=1, label='Random baseline')
ax.set_xlim(0.45, 1.0)
ax.set_xlabel('Test Accuracy')
ax.set_title('Sentiment Analysis — All Methods (IMDB, n=5,000 test)', fontsize=13)
for bar, acc in zip(bars, all_results_df['accuracy']):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
            f'{acc:.4f}', va='center', fontsize=9)

from matplotlib.patches import Patch
legend = [
    Patch(color='#3498db', label='Sparse supervised'),
    Patch(color='#e74c3c', label='Dense supervised (W2V)'),
    Patch(color='#95a5a6', label='Zero-shot (no labels)'),
]
ax.legend(handles=legend, fontsize=10)
plt.tight_layout()
plt.show()

---
## Summary

### Results interpretation

| Method | Type | Typical accuracy | Key property |
|---|---|---|---|
| **BERT [CLS] Contextual** | Dense | ~92–94% | Context-aware; attention solves negation |
| **BM25** | Sparse | ~90–91% | Best sparse — length normalisation helps |
| **TF-IDF** | Sparse | ~89–90% | Strong baseline |
| **BERT Input Embeddings** | Dense static | ~87–89% | Same lookup table as W2V but 768-dim + subwords |
| **Bag-of-Words** | Sparse | ~87–89% | Surprisingly competitive |
| **W2V Min/Mean/Max** | Dense | ~86–88% | Best W2V — min pooling catches negations |
| **W2V TF-IDF Weighted** | Dense | ~85–87% | Marginal gain over plain mean |
| **W2V Mean** | Dense | ~83–86% | Baseline dense |

### The static vs contextual gap (BERT Input vs BERT [CLS])

Both use the same 768-dim space and the same tokenizer. The only difference is whether the 12 attention layers run.

- **BERT Input** (~87–89%): static lookup — "not good" still averages to something similar to "good"
- **BERT [CLS]** (~92–94%): each token's vector is reshaped by its context — "not" genuinely modifies "good"

The gap between them is the pure value added by self-attention.

### Why sparse still beats BERT input embeddings

The input embedding matrix is trained to be a good **initialisation for the attention layers**, not a standalone representation. It encodes token identity efficiently for the transformer to work with, but is not optimised for direct mean-pooling over sentences. Sparse methods keep each sentiment word as its own independent feature — the classifier can assign exactly the right weight to "masterpiece" or "waste" without semantic averaging diluting the signal.

### What this tells us about transformer embeddings

```
token IDs → [static input matrix]  ─── already beats W2V (higher dim, subword coverage)
                  ↓
          [12 × self-attention]    ─── adds context, solves negation, polysemy
                  ↓
          [CLS output vector]      ─── best single-vector sentence representation
```

Each stage adds something measurable. The input matrix alone is a strong static baseline; the attention layers on top close the remaining gap to sparse methods and exceed them.